In [6]:
import pandas as pd
import numpy as np
import os

# 1. SETUP - File Paths
INPUT_FOLDER = '../Input_Files/'
OUTPUT_FOLDER = '../Output_files/'

hosp_file = os.path.join(INPUT_FOLDER, 'Hospital_General_Information.csv')
bill_file = os.path.join(INPUT_FOLDER, 'Medicare_IP_Hospitals_by_Provider_and_Service_2023.csv')

# 2. DICTIONARY - State & Territory Mapping
state_map = {
    'AK': 'Alaska', 'AL': 'Alabama', 'AR': 'Arkansas', 'AZ': 'Arizona', 'CA': 'California',
    'CO': 'Colorado', 'CT': 'Connecticut', 'DC': 'District of Columbia', 'DE': 'Delaware',
    'FL': 'Florida', 'GA': 'Georgia', 'HI': 'Hawaii', 'IA': 'Iowa', 'ID': 'Idaho',
    'IL': 'Illinois', 'IN': 'Indiana', 'KS': 'Kansas', 'KY': 'Kentucky', 'LA': 'Louisiana',
    'MA': 'Massachusetts', 'MD': 'Maryland', 'ME': 'Maine', 'MI': 'Michigan', 'MN': 'Minnesota',
    'MO': 'Missouri', 'MS': 'Mississippi', 'MT': 'Montana', 'NC': 'North Carolina',
    'ND': 'North Dakota', 'NE': 'Nebraska', 'NH': 'New Hampshire', 'NJ': 'New Jersey',
    'NM': 'New Mexico', 'NV': 'Nevada', 'NY': 'New York', 'OH': 'Ohio', 'OK': 'Oklahoma',
    'OR': 'Oregon', 'PA': 'Pennsylvania', 'RI': 'Rhode Island', 'SC': 'South Carolina',
    'SD': 'South Dakota', 'TN': 'Tennessee', 'TX': 'Texas', 'UT': 'Utah', 'VA': 'Virginia',
    'VT': 'Vermont', 'WA': 'Washington', 'WI': 'Wisconsin', 'WV': 'West Virginia', 'WY': 'Wyoming',
    'PR': 'Puerto Rico', 'VI': 'Virgin Islands', 'GU': 'Guam', 'AS': 'American Samoa', 'MP': 'Northern Mariana Islands'
}

territories = ['PR', 'VI', 'GU', 'AS', 'MP']

# 3. LOAD DATA
print("🔄 Step 1: Loading raw data...")
hosp_df = pd.read_csv(hosp_file, encoding='latin-1')
bill_df = pd.read_csv(bill_file, encoding='latin-1')

# 4. DIM_LOCATIONS (Geography Dimension)
print("📍 Step 2: Creating Locations table (ID at start)...")
locations = hosp_df[['City/Town', 'State', 'ZIP Code', 'County/Parish']].drop_duplicates().reset_index(drop=True)
locations.columns = ['city', 'state', 'zip', 'county']

# Generate and position location_id at the start
locations['location_id'] = range(10001, 10001 + len(locations))
locations['state_full'] = locations['state'].map(state_map)
locations['country'] = locations['state'].apply(lambda x: 'US Territory' if x in territories else 'United States')

# Final Column Order for Locations
locations = locations[['location_id', 'city', 'state', 'state_full', 'zip', 'county', 'country']]


# 5. HOSPITALS (Hospital Dimension)
print("📑 Step 3: Creating Hospitals table (Fixed Merge and Column Order)...")

# FIX: We use left_on (original names) and right_on (new names) to fix the KeyError
hosp_merged = hosp_df.merge(
    locations, 
    left_on=['City/Town', 'State', 'ZIP Code', 'County/Parish'],
    right_on=['city', 'state', 'zip', 'county'],
    how='left'
)

# Final Column Order: facility_id FIRST, location_id SECOND, then address and others
hospitals = hosp_merged[['Facility ID', 'location_id', 'Facility Name', 'Hospital Type', 'Hospital Ownership', 'Address','Emergency Services','Hospital overall rating']].copy()
hospitals.columns = ['facility_id', 'location_id', 'hospital_name', 'type', 'ownership', 'address','emergency_service','rating']

# Data Cleaning
hospitals['rating'] = pd.to_numeric(hospitals['rating'].replace('Not Available', np.nan))
hospitals['facility_id'] = hospitals['facility_id'].astype(str).str.zfill(6)

# 6. REF_DRG (The Procedure Reference)
print("🩺 Step 4: Creating DRG reference table...")
ref_drg = bill_df[['DRG_Cd', 'DRG_Desc']].drop_duplicates()
ref_drg.columns = ['drg_code', 'drg_description']


# 7. BILLING (Fact Table)
print("💰 Step 5: Creating Billing fact table...")
billing = bill_df[['Rndrng_Prvdr_CCN', 'DRG_Cd', 'Tot_Dschrgs', 'Avg_Submtd_Cvrd_Chrg', 'Avg_Tot_Pymt_Amt', 'Avg_Mdcr_Pymt_Amt']].copy()
billing.columns = ['facility_id', 'drg_code', 'total_discharges', 'avg_submitted_charges', 'avg_total_payment', 'avg_medicare_payment']
billing['facility_id'] = billing['facility_id'].astype(str).str.zfill(6)

print("\n🚀 SUCCESS: The KeyError is resolved and files are perfectly formatted!")

🔄 Step 1: Loading raw data...
📍 Step 2: Creating Locations table (ID at start)...
📑 Step 3: Creating Hospitals table (Fixed Merge and Column Order)...
🩺 Step 4: Creating DRG reference table...
💰 Step 5: Creating Billing fact table...

🚀 SUCCESS: The KeyError is resolved and files are perfectly formatted!


Checkig Duplicates in each file

In [7]:
print(hospitals.duplicated().sum())
print(locations.duplicated().sum())
print(ref_drg.duplicated().sum())
print(billing.duplicated().sum())

0
0
0
0


checking for nulls:

In [8]:
hospitals.isnull().sum()

facility_id             0
location_id             0
hospital_name           0
type                    0
ownership               0
address                 0
emergency_service       0
rating               2560
dtype: int64

Keep ratings as NULL because it correctly represents missing data while preserving the column as numeric for accurate calculations and analysis.

In [9]:
locations.isnull().sum()

location_id    0
city           0
state          0
state_full     0
zip            0
county         0
country        0
dtype: int64

In [10]:
ref_drg.isnull().sum()

drg_code           0
drg_description    0
dtype: int64

In [11]:
billing.isnull().sum()

facility_id              0
drg_code                 0
total_discharges         0
avg_submitted_charges    0
avg_total_payment        0
avg_medicare_payment     0
dtype: int64

Checking Datatypes

In [12]:
print("Data types in Hospital table:\n",hospitals.dtypes)
print("\nData types in locations table:\n",hospitals.dtypes)
print("\nData types in ref_drg table:\n",hospitals.dtypes)
print("\nData types in billing table:\n",billing.dtypes)

Data types in Hospital table:
 facility_id           object
location_id            int64
hospital_name         object
type                  object
ownership             object
address               object
emergency_service     object
rating               float64
dtype: object

Data types in locations table:
 facility_id           object
location_id            int64
hospital_name         object
type                  object
ownership             object
address               object
emergency_service     object
rating               float64
dtype: object

Data types in ref_drg table:
 facility_id           object
location_id            int64
hospital_name         object
type                  object
ownership             object
address               object
emergency_service     object
rating               float64
dtype: object

Data types in billing table:
 facility_id               object
drg_code                   int64
total_discharges           int64
avg_submitted_charges    float64
avg_

In [13]:
hospitals['rating'] = pd.to_numeric(
    hospitals['rating'].astype(str).str.strip(),
    errors='coerce'
).astype('Int64')

Seeing the data to check what it contain and if we need more handling

In [14]:
hospitals.head()

,facility_id,location_id,hospital_name,type,ownership,address,emergency_service,rating
0,010001,10001,SOUTHEAST HEALTH MEDICAL CENTER,Acute Care Hospitals,Government - Hospital District or Authority,1108 ROSS CLARK CIRCLE,Yes,4
1,010005,10002,MARSHALL MEDICAL CENTERS,Acute Care Hospitals,Government - Hospital District or Authority,2505 U S HIGHWAY 431 NORTH,Yes,3
2,010006,10003,NORTH ALABAMA MEDICAL CENTER,Acute Care Hospitals,Proprietary,1701 VETERANS DRIVE,Yes,2
3,010007,10004,MIZELL MEMORIAL HOSPITAL,Acute Care Hospitals,Voluntary non-profit - Private,702 N MAIN ST,Yes,1
4,010008,10005,CRENSHAW COMMUNITY HOSPITAL,Acute Care Hospitals,Proprietary,101 HOSPITAL CIRCLE,Yes,<NA>


In [15]:
locations.head()

,location_id,city,state,state_full,zip,county,country
0,10001,DOTHAN,AL,Alabama,36301,HOUSTON,United States
1,10002,BOAZ,AL,Alabama,35957,MARSHALL,United States
2,10003,FLORENCE,AL,Alabama,35630,LAUDERDALE,United States
3,10004,OPP,AL,Alabama,36467,COVINGTON,United States
4,10005,LUVERNE,AL,Alabama,36049,CRENSHAW,United States


In [16]:
ref_drg.head()

,drg_code,drg_description
0,3,ECMO OR TRACHEOSTOMY WITH MV >96 HOURS OR PRIN...
1,23,CRANIOTOMY WITH MAJOR DEVICE IMPLANT OR ACUTE ...
2,24,CRANIOTOMY WITH MAJOR DEVICE IMPLANT OR ACUTE ...
3,25,CRANIOTOMY AND ENDOVASCULAR INTRACRANIAL PROCE...
4,38,EXTRACRANIAL PROCEDURES WITH CC


In [17]:
billing.head()

,facility_id,drg_code,total_discharges,avg_submitted_charges,avg_total_payment,avg_medicare_payment
0,010001,3,14,663764.35714,120219.928570,115544.142860
1,010001,23,26,180980.88462,37321.038462,35261.807692
2,010001,24,12,105824.33333,26936.666667,25048.916667
3,010001,25,16,242539.50000,34745.375000,32438.625000
4,010001,38,11,122741.18182,14999.818182,9579.363636


while importing these drg_description using mysql query, error was coming as this contains ',' in description and then it was confused where the column data ends. if you are importing csv file then no need of this step

In [18]:
ref_drg['drg_description'] = ref_drg['drg_description'].str.replace(',', '', regex=False)

Creating csv files out of dataframes

In [19]:

hospitals.to_csv(os.path.join(OUTPUT_FOLDER, 'hospitals.csv'), index=False)
locations.to_csv(os.path.join(OUTPUT_FOLDER, 'locations.csv'), index=False)
ref_drg.to_csv(os.path.join(OUTPUT_FOLDER, 'ref_drg.csv'), index=False)
billing.to_csv(os.path.join(OUTPUT_FOLDER, 'billing.csv'), index=False)

Four files are ready for Analysis